# Component resolution: which sublayer and which heads write the CoT-injection override

Parent result (M1): attacker-authored reasoning prefilled into the `<think>` trace of `DeepSeek-R1-Distill-Qwen-1.5B` flips its multiple-choice answer to a target letter. Full-residual patching at the answer position put the override in a narrow mid-late band, layers ~17–22 of 28. That tells us *where* it lands in the residual, not *which sublayer* wrote it there. This notebook splits that write apart.

The plan:
- **Validation (gate)** — does the harness reproduce the parent full-residual curve? Expect ~0 through layer 16, a rise across 17–21, plateau near 1.
- **A** — patch `attn_out[L]` vs `mlp_out[L]` separately; whichever climbs across 17–22 is the writer.
- **B** — if attention wins, rank the heads at the peak layer.
- **C** — are the top heads sufficient (patch alone → flip) and necessary (winner beats loser at the commit layer)?
- **Stretch** — can those same heads steer a *different* item to a letter we pick?

The two outcomes mean different things. Attention-written ⇒ induction/retrieval heads copying the planted conclusion (Olsson et al. 2022). MLP-written ⇒ a mid-layer association being overwritten (ROME, Meng et al. 2022). The mechanism is the result, not the layer index.

Metric is unchanged from the parent run so the numbers stay comparable: normalized recovery `(patched_ld − clean_ld) / (inj_ld − clean_ld)` in target-minus-baseline logit space. Filters: capturable (clean ≠ target) and `|effect| ≥ 1`. Greedy decoding, bootstrap 1000 resamples, seed 0. Input is `capturable_traces.jsonl`, one record per capturable item.

## 1 · Setup & config

Upload `capturable_traces.jsonl`, install deps, grab a GPU, load the model in bf16 (fp16 gives NaN logits on this one). Architecture is read off `model.config` rather than hardcoded.

In [ ]:
# Upload capturable_traces.jsonl (skipped if already next to the notebook)
from pathlib import Path
DATA_PATH = "capturable_traces.jsonl"
if not Path(DATA_PATH).exists():
    try:
        from google.colab import files
        up = files.upload()
        if DATA_PATH not in up:
            print(f"WARNING: expected {DATA_PATH}, got {list(up)}")
    except Exception as e:
        print(f"Not on Colab / no upload widget; place {DATA_PATH} next to the notebook. ({e})")
print("data present:", Path(DATA_PATH).exists())

In [ ]:
import sys, subprocess
def _pip(*pkgs): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
_pip("transformers>=4.44", "accelerate", "matplotlib")
print("install step done")

In [ ]:
import re, json, hashlib
from dataclasses import dataclass
from typing import List, Optional, Dict, Tuple
from collections import Counter
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

CONFIG = dict(
    MODEL_ID   = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    DTYPE      = torch.bfloat16,      # fp16 -> NaN logits on this model
    DEVICE     = "cuda",
    LETTERS    = "ABCD",
    MAX_NEW    = 900,                 # greedy gen budget (matches parent)
    SEED       = 0,
    WINDOW     = (17, 22),            # parent localization window (for shading + the A decision)
    EFFECT_MIN = 1.0,                 # |inj_ld - clean_ld| filter (recovery ill-defined below this)
    N_BOOT     = 1000,                # bootstrap resamples, seed 0 (parent setting)
    DEV_N      = 24,                  # dev subset size for fast iteration
)
assert torch.cuda.is_available(), "CUDA GPU required (bf16 on CUDA; fp16 NaNs on this model)."
torch.manual_seed(CONFIG["SEED"]); np.random.seed(CONFIG["SEED"])
print(CONFIG["DEVICE"], CONFIG["MODEL_ID"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_ID"])
model = AutoModelForCausalLM.from_pretrained(           # AutoModelForCausalLM (has LM head)
    CONFIG["MODEL_ID"], torch_dtype=CONFIG["DTYPE"]).to(CONFIG["DEVICE"]).eval()

# Read architecture from config — do NOT hardcode.
LAYERS   = model.config.num_hidden_layers               # 28
N_HEADS  = model.config.num_attention_heads             # 12 query heads
D_MODEL  = model.config.hidden_size                     # 1536
HEAD_DIM = D_MODEL // N_HEADS                            # 128
N_KV     = getattr(model.config, "num_key_value_heads", N_HEADS)   # 2 (GQA)
assert N_HEADS * HEAD_DIM == D_MODEL, "o_proj input is not query-head aligned"
print(f"layers={LAYERS}  d_model={D_MODEL}  q_heads={N_HEADS}  kv_heads={N_KV}  head_dim={HEAD_DIM}")
print(f"GQA={N_KV != N_HEADS} — the o_proj INPUT is organized by QUERY heads, so a head slice "
      f"[h*{HEAD_DIM}:(h+1)*{HEAD_DIM}] is clean to patch.")

## 2 · Core functions

Two blocks. The first is lifted verbatim from the parent notebook — answer-locating, tokenization, full-residual patching — so the numbers below line up with M1 exactly. The second adds the new piece: component- and head-level capture/patch via forward hooks.

If you'd rather keep the module split, drop these two cells into `component_resolution.py` and `%run` it. The names don't change.

In [ ]:
# --- parent helpers, reused VERBATIM (do not change: comparability with M1 depends on these) ---
@dataclass
class Item:                          # the prompt's `Item`; fields match capturable_traces.jsonl
    question: str
    options: List[str]
    target_letter: str
    injected_think: str
    clean_baseline_letter: Optional[str] = None
    length: Optional[int] = None
    family: Optional[str] = None

def format_question(item):
    opts = "\n".join(f"{L}. {t}" for L, t in zip(CONFIG["LETTERS"], item.options))
    return f"{item.question}\n\n{opts}\n\nAnswer with the single capital letter of the correct option."

def _prefix_ids_with_think(item):
    msgs = [{"role": "user", "content": format_question(item)}]
    enc  = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True)
    if hasattr(enc, "ids"): ids = list(enc.ids)
    elif hasattr(enc, "input_ids") or isinstance(enc, dict):
        ii = enc["input_ids"]; ids = list(ii[0]) if (len(ii) and isinstance(ii[0], (list, tuple))) else list(ii)
    else: ids = list(enc)
    if "<think>" not in tokenizer.decode(ids[-8:]):
        ids = ids + tokenizer.encode("<think>\n", add_special_tokens=False)
    return ids

def letter_token_ids(letters):
    out = {}
    for L in letters:
        enc = tokenizer.encode(L, add_special_tokens=False)
        assert len(enc) == 1, f"{L!r} not single token: {enc}"
        out[L] = enc[0]
    return out
LETTER_ID = letter_token_ids(CONFIG["LETTERS"])

@torch.inference_mode()
def generate_and_locate(item, injected):
    # greedy-generate, then regex-locate the commit token; the answer POSITION is the token
    # whose next-token logits are the boxed answer (so we read prefix[:, -1, :]).
    header_ids = _prefix_ids_with_think(item)
    think_ids  = tokenizer.encode(item.injected_think, add_special_tokens=False) if injected and item.injected_think else []
    close_ids  = tokenizer.encode("\n</think>\n\n", add_special_tokens=False)
    all_ids    = header_ids + think_ids + close_ids
    input_ids  = torch.tensor([all_ids], device=CONFIG["DEVICE"])
    out = model.generate(input_ids, max_new_tokens=CONFIG["MAX_NEW"], do_sample=False,
                         pad_token_id=tokenizer.eos_token_id, attention_mask=torch.ones_like(input_ids))
    full_ids, generated_ids = out[0].tolist(), out[0, len(all_ids):].tolist()
    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    _PATTERNS = [r'\*\*([ABCD])\.', r'\*\*([ABCD])\*\*', r'\\boxed\{([ABCD])\}?',
                 r'\\text\{([ABCD])\}', r'[Aa]nswer\s+is\s*:?\s*\**([ABCD])\b', r'\(\\boxed\{([ABCD])\}']
    hits = []
    for pat in _PATTERNS:
        for m in re.finditer(pat, gen_text): hits.append((m.start(), m.group(1).upper()))
    if not hits:
        num = re.search(r'\\boxed\{([^A-Da-d{}][^{}]{0,30})\}', gen_text)
        return (None, ("numeric" if num else "truncated"), (0, 0))
    answer_letter = sorted(hits)[-1][1]; letter_tok = LETTER_ID[answer_letter]
    idx = None
    for i in range(len(generated_ids) - 1, -1, -1):
        if generated_ids[i] == letter_tok and any(c in tokenizer.decode(generated_ids[max(0,i-6):i]) for c in ("**","boxed","{")):
            idx = i; break
    if idx is None:
        for i in range(len(generated_ids) - 1, -1, -1):
            if generated_ids[i] == letter_tok: idx = i; break
    if idx is None: return (None, "no_token", (0, 0))
    prefix = torch.tensor([full_ids[:len(all_ids)+idx]], device=CONFIG["DEVICE"])
    return prefix, answer_letter, (len(header_ids), len(header_ids)+len(think_ids))

def _layers(): return model.model.layers

@torch.inference_mode()
def last_logits(input_ids): return model(input_ids).logits[:, -1, :]

def logit_diff(logits, tgt, base): return float(logits[0, LETTER_ID[tgt]] - logits[0, LETTER_ID[base]])
def recovery(p, c, i, eps=1e-6): return (p - c) / ((i - c) + eps)
def letter_argmax(logits):
    ids = [LETTER_ID[L] for L in CONFIG["LETTERS"]]
    p = torch.softmax(logits[0, ids].float(), -1)
    return CONFIG["LETTERS"][int(p.argmax())]

In [ ]:
# --- NEW: component- & head-level capture/patch (HF forward hooks) ---
# Qwen2 decoder layer:  h = h + self_attn(ln1(h));  h = h + mlp(ln2(h))
#   attn_out[L] = self_attn forward output      mlp_out[L] = mlp forward output
#   o_proj INPUT = per-head attention concat [1, N_HEADS*HEAD_DIM] -> head h is slice [h*HD:(h+1)*HD]

@torch.inference_mode()
def capture_injected(input_ids, pos=-1):
    # one forward over a prefix; grab, at position `pos` and per layer:
    #   'resid' full residual (layer output), 'attn' self_attn out, 'mlp' mlp out,
    #   'ohead' o_proj input (per-head concat) used for head patching.
    store = {"resid": {}, "attn": {}, "mlp": {}, "ohead": {}}
    handles = []
    def mk_out(bucket, L):
        def hook(mod, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            store[bucket][L] = h[:, pos, :].detach().clone()
        return hook
    def mk_pre(L):
        def hook(mod, inp):
            store["ohead"][L] = inp[0][:, pos, :].detach().clone()
        return hook
    for L, layer in enumerate(_layers()):
        handles.append(layer.register_forward_hook(mk_out("resid", L)))
        handles.append(layer.self_attn.register_forward_hook(mk_out("attn", L)))
        handles.append(layer.mlp.register_forward_hook(mk_out("mlp", L)))
        handles.append(layer.self_attn.o_proj.register_forward_pre_hook(mk_pre(L)))
    try: model(input_ids)
    finally:
        for h in handles: h.remove()
    return store

def capture_resid(input_ids, pos=-1):     # parity alias with the parent notebook
    return capture_injected(input_ids, pos=pos)["resid"]

@torch.inference_mode()
def patch_resid_ld(clean_prefix, L, donor_vec, tgt, base, pos=-1):
    def hook(mod, inp, out):
        if isinstance(out, tuple):
            h = out[0].clone(); h[:, pos, :] = donor_vec.to(h.dtype); return (h,) + tuple(out[1:])
        h = out.clone(); h[:, pos, :] = donor_vec.to(h.dtype); return h
    handle = _layers()[L].register_forward_hook(hook)
    try: return logit_diff(model(clean_prefix).logits[:, -1, :], tgt, base)
    finally: handle.remove()

@torch.inference_mode()
def patch_component_ld(clean_prefix, L, component, donor_vec, tgt, base, pos=-1):
    # component in {'attn','mlp'} — overwrite that sublayer's marginal write at `pos`.
    mod = _layers()[L].self_attn if component == "attn" else _layers()[L].mlp
    def hook(m, inp, out):
        if isinstance(out, tuple):
            h = out[0].clone(); h[:, pos, :] = donor_vec.to(h.dtype); return (h,) + tuple(out[1:])
        h = out.clone(); h[:, pos, :] = donor_vec.to(h.dtype); return h
    handle = mod.register_forward_hook(hook)
    try: return logit_diff(model(clean_prefix).logits[:, -1, :], tgt, base)
    finally: handle.remove()

@torch.inference_mode()
def patch_heads_ld(clean_prefix, L, heads, donor_ohead, tgt, base, pos=-1):
    # overwrite the o_proj-input slice(s) for `heads` at `pos` with the donor's head value(s).
    heads = [heads] if isinstance(heads, int) else list(heads)
    def pre_hook(m, inp):
        x = inp[0].clone()
        for h in heads:
            sl = slice(h*HEAD_DIM, (h+1)*HEAD_DIM)
            x[:, pos, sl] = donor_ohead[:, sl].to(x.dtype)
        return (x,) + tuple(inp[1:])
    handle = _layers()[L].self_attn.o_proj.register_forward_pre_hook(pre_hook)
    try: return logit_diff(model(clean_prefix).logits[:, -1, :], tgt, base)
    finally: handle.remove()

def patch_head_ld(clean_prefix, L, head, donor_ohead, tgt, base, pos=-1):
    return patch_heads_ld(clean_prefix, L, [head], donor_ohead, tgt, base, pos=pos)

# --- sweeps over a prepared per-item context ---
def run_full_resid_sweep(ctx, layers=None):
    layers = range(LAYERS) if layers is None else layers
    return [recovery(patch_resid_ld(ctx["c_prefix"], L, ctx["store"]["resid"][L], ctx["tgt"], ctx["base"]),
                     ctx["c_ld"], ctx["i_ld"]) for L in layers]

def run_component_sweep(ctx, layers=None):
    layers = range(LAYERS) if layers is None else layers
    out = {"attn": [], "mlp": []}
    for comp in ("attn", "mlp"):
        out[comp] = [recovery(patch_component_ld(ctx["c_prefix"], L, comp, ctx["store"][comp][L],
                                                 ctx["tgt"], ctx["base"]), ctx["c_ld"], ctx["i_ld"])
                     for L in layers]
    return out

def run_head_sweep(ctx, L):
    donor = ctx["store"]["ohead"][L]
    return [recovery(patch_heads_ld(ctx["c_prefix"], L, [h], donor, ctx["tgt"], ctx["base"]),
                     ctx["c_ld"], ctx["i_ld"]) for h in range(N_HEADS)]

# --- small utilities ---
def commit_layer(curve, thresh=0.5):
    for L, v in enumerate(curve):
        if v >= thresh: return L
    return None

def bootstrap_band(arr, n=None, seed=0):
    # arr: [n_items, n_layers]; returns (mean, lo95, hi95) over n resamples (deterministic).
    n = CONFIG["N_BOOT"] if n is None else n
    arr = np.asarray(arr, dtype=float); m = arr.mean(0)
    if len(arr) < 2: return m, m, m
    rng = np.random.RandomState(seed)
    boot = np.array([arr[rng.randint(0, len(arr), len(arr))].mean(0) for _ in range(n)])
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return m, lo, hi
print("core functions ready")

## 3 · Data loading + per-item contexts

`load_items()` reads the JSONL into `Item`s and re-checks the capturable filter (clean ≠ target). `prepare_item()` does the slow part once per item: greedy-generate clean and injected, locate the answer positions, and capture the injected donor store (full residual + attn + mlp + per-head). Located prefixes and endpoints are cached to `component_prep.jsonl`, so a reconnect is cheap — prefixes rebuild from token-ids, only the cheap capture re-runs. Contexts live in `CTX` so every phase shares one capture.

The `|effect| ≥ 1` filter runs after prep, since it needs the endpoints. We keep a ~24-item dev subset for fast iteration alongside the full set.

In [ ]:
def load_items(path):
    return [Item(**json.loads(l)) for l in Path(path).read_text().splitlines() if l.strip()]

def item_key(item):
    return hashlib.md5(f"{item.family}|{item.length}|{item.question}".encode()).hexdigest()[:16]

PREP_FILE = "component_prep.jsonl"      # caches located prefixes + endpoints (the costly generations)
_PREP_DISK = {}
if Path(PREP_FILE).exists():
    for line in open(PREP_FILE):
        if line.strip():
            r = json.loads(line); _PREP_DISK[r["key"]] = r
    print(f"prep cache: {len(_PREP_DISK)} items on disk")

CTX = {}                                # live contexts (tensors + donor store), keyed by item_key

def _save_skip(k, reason, fh):
    rec = {"key": k, "skip": reason}; _PREP_DISK[k] = rec
    if fh is not None: fh.write(json.dumps(rec) + "\n"); fh.flush()
    return {"skip": reason, "key": k}

def prepare_item(item, _fh=None):
    # return a live context dict, or {'skip': reason}. Reuses the disk cache to skip regeneration;
    # always re-captures the (cheap) donor store so the phases share it.
    k = item_key(item)
    rec = _PREP_DISK.get(k)
    if rec is not None and "skip" in rec:
        return {"skip": rec["skip"], "key": k}
    if rec is not None:
        c_prefix = torch.tensor([rec["c_ids"]], device=CONFIG["DEVICE"])
        i_prefix = torch.tensor([rec["i_ids"]], device=CONFIG["DEVICE"])
        base, tgt = rec["base"], rec["target"]
        c_ld, i_ld = rec["clean_ld"], rec["inject_ld"]
        c_letter, i_letter = rec["clean_letter"], rec["inject_letter"]
    else:
        c_prefix, c_letter, _ = generate_and_locate(item, injected=False)
        if c_prefix is None: return _save_skip(k, f"clean_{c_letter}", _fh)
        i_prefix, i_letter, _ = generate_and_locate(item, injected=True)
        if i_prefix is None: return _save_skip(k, f"inject_{i_letter}", _fh)
        base = item.clean_baseline_letter or c_letter
        tgt  = item.target_letter
        if base == tgt: return _save_skip(k, "not_capturable", _fh)
        c_ld = logit_diff(last_logits(c_prefix), tgt, base)
        i_ld = logit_diff(last_logits(i_prefix), tgt, base)
        rec = dict(key=k, family=item.family, length=item.length, base=base, target=tgt,
                   clean_letter=c_letter, inject_letter=i_letter, clean_ld=c_ld, inject_ld=i_ld,
                   effect=i_ld - c_ld, flipped=bool(i_letter == tgt),
                   c_ids=c_prefix[0].tolist(), i_ids=i_prefix[0].tolist())
        _PREP_DISK[k] = rec
        if _fh is not None: _fh.write(json.dumps(rec) + "\n"); _fh.flush()
    store = capture_injected(i_prefix, pos=-1)
    ctx = dict(key=k, family=item.family, length=item.length, base=base, tgt=tgt,
               c_prefix=c_prefix, i_prefix=i_prefix, c_ld=c_ld, i_ld=i_ld, effect=i_ld - c_ld,
               c_letter=c_letter, i_letter=i_letter, flipped=bool(i_letter == tgt), store=store)
    CTX[k] = ctx
    return ctx

In [ ]:
raw_items = load_items(DATA_PATH)
items = [it for it in raw_items if (it.clean_baseline_letter or "") != it.target_letter]   # capturable
print(f"loaded {len(raw_items)} records; capturable (clean != target): {len(items)}")
print("by (family,length):", dict(Counter((it.family, it.length) for it in items)))

prepared, skipped = [], []
with open(PREP_FILE, "a") as fh:
    for j, it in enumerate(items):
        ctx = prepare_item(it, _fh=fh)
        tag = f"[{j+1:>3}/{len(items)}] {it.family}@{it.length}"
        if "skip" in ctx:
            skipped.append((it, ctx["skip"])); print(tag, "skip:", ctx["skip"])
        else:
            prepared.append(ctx); print(tag, f"flip={ctx['flipped']} effect={ctx['effect']:+.2f}")
        items[j]._key = ctx.get("key")     # remember for the redirect pairing

substantial = [c for c in prepared if abs(c["effect"]) >= CONFIG["EFFECT_MIN"]]
print(f"\ncounts — total:{len(items)}  prepared:{len(prepared)}  "
      f"effect-substantial(|eff|>={CONFIG['EFFECT_MIN']}):{len(substantial)}  skipped:{len(skipped)}")

DEV  = substantial[:CONFIG["DEV_N"]]     # fast iteration
FULL = substantial                       # the comparable ~70-item set
assert len(FULL) >= 2, "no effect-substantial items — check the data / generation step"
print(f"dev subset n={len(DEV)}   full set n={len(FULL)}")

## 4 · Validation (gate)

Patch the whole residual at the answer position, layer by layer. Recovery should sit at ~0 through layer 16, rise across 17–21, and plateau near 1. Two sanity checks alongside it: patching a layer with the item's own clean value should be a no-op (~0), and patching the injected value at a late layer should recover most of the effect.

This is a gate. If the curve doesn't reproduce, a wrong answer-position or tokenization is silently poisoning everything downstream — stop and fix that before reading on.

In [ ]:
full_curves = np.array([run_full_resid_sweep(c) for c in DEV])
m_full, lo_full, hi_full = bootstrap_band(full_curves, seed=0)

plt.figure(figsize=(9, 4))
plt.plot(range(LAYERS), m_full, marker="o", ms=3, label=f"full residual (n={len(DEV)})")
plt.fill_between(range(LAYERS), lo_full, hi_full, alpha=0.2)
plt.axhline(0, ls="--", lw=.8, c="gray"); plt.axhline(1, ls="--", lw=.8, c="gray")
plt.axvspan(CONFIG["WINDOW"][0], CONFIG["WINDOW"][1], alpha=0.08, color="red")
plt.xlabel("layer"); plt.ylabel("normalized recovery")
plt.title("Validation: full-residual recovery (expect rise across 17–21, plateau ~1)")
plt.legend(); plt.tight_layout(); plt.savefig("fig_validation_fullresid.png", dpi=130); plt.show()

early = float(np.mean(m_full[:17]))         # layers 0–16: pre-window, should be flat ~0
late  = float(np.mean(m_full[23:]))         # layers 23–27: post-commit plateau (the rise crosses
                                            # the window at L22 — checking the plateau here, after
                                            # the commit jump, rather than mid-rise, is what makes
                                            # "plateaus near 1" a meaningful, non-flaky assertion)
print(f"mean recovery layers 0–16  = {early:+.2f}  (expect ~0)")
print(f"mean recovery layers 23–27 = {late:+.2f}  (expect ~1, post-commit plateau)")

probe = DEV[0]
clean_store = capture_resid(probe["c_prefix"], pos=-1)
noop = recovery(patch_resid_ld(probe["c_prefix"], 20, clean_store[20], probe["tgt"], probe["base"]),
                probe["c_ld"], probe["i_ld"])
inj20 = recovery(patch_resid_ld(probe["c_prefix"], 20, probe["store"]["resid"][20], probe["tgt"], probe["base"]),
                 probe["c_ld"], probe["i_ld"])
print(f"hook-sanity  no-op@L20 = {noop:+.3f} (expect ~0)   injected@L20 = {inj20:+.3f} (expect close to full curve)")

assert abs(noop) < 0.05, f"no-op patch should be ~0, got {noop}"
assert early < 0.25,     f"early layers should be ~0, got {early}"
assert late  > 0.6,      f"late window should plateau near 1, got {late}"
VALIDATION_OK = True
print("\nVALIDATION PASSED — proceeding to the component curves.")

## 5 · Phase A — attention vs MLP

Patch `attn_out[L]` and `mlp_out[L]` separately across every layer (dev first, then full). Whichever curve climbs toward 1 across 17–22 is the writer. The cell prints the call — attention-written, MLP-written, or mixed — plus the peak layer.

In [ ]:
assert globals().get("VALIDATION_OK"), "Run & pass the Validation cell before Phase A."

def component_sweep_set(ctxs):
    attn, mlp = [], []
    for c in ctxs:
        s = run_component_sweep(c); attn.append(s["attn"]); mlp.append(s["mlp"])
    return np.array(attn), np.array(mlp)

_a, _m = component_sweep_set(DEV);  print(f"dev Phase A done (n={len(DEV)})")
attn_arr, mlp_arr = component_sweep_set(FULL); print(f"full Phase A done (n={len(FULL)})")

ma, loa, hia = bootstrap_band(attn_arr, seed=0)
mm, lom, him = bootstrap_band(mlp_arr,  seed=0)

plt.figure(figsize=(10, 5))
plt.plot(range(LAYERS), ma, marker="o", ms=3, color="C0", label=f"attn_out (n={len(FULL)})")
plt.fill_between(range(LAYERS), loa, hia, alpha=0.18, color="C0")
plt.plot(range(LAYERS), mm, marker="s", ms=3, color="C1", label=f"mlp_out (n={len(FULL)})")
plt.fill_between(range(LAYERS), lom, him, alpha=0.18, color="C1")
plt.plot(range(LAYERS), m_full, ls=":", c="gray", label="full residual (validation)")
plt.axhline(0, ls="--", lw=.8, c="gray"); plt.axhline(1, ls="--", lw=.8, c="gray")
plt.axvspan(CONFIG["WINDOW"][0], CONFIG["WINDOW"][1], alpha=0.08, color="red")
plt.xlabel("layer"); plt.ylabel("normalized recovery")
plt.title("Phase A — attention vs MLP marginal write (17–22 shaded)")
plt.legend(); plt.tight_layout(); plt.savefig("fig_phaseA_attn_vs_mlp.png", dpi=130); plt.show()

w0, w1 = CONFIG["WINDOW"]; win = slice(w0, w1 + 1)
attn_win = float(np.nanmean(ma[win])); mlp_win = float(np.nanmean(mm[win]))
peak_layer_attn = int(w0 + int(np.nanargmax(ma[win])))
peak_layer_mlp  = int(w0 + int(np.nanargmax(mm[win])))
print(f"window {w0}-{w1} mean recovery:  attn={attn_win:+.2f}  mlp={mlp_win:+.2f}")
print(f"peak-in-window:  attn@L{peak_layer_attn} ({ma[peak_layer_attn]:+.2f})  "
      f"mlp@L{peak_layer_mlp} ({mm[peak_layer_mlp]:+.2f})")

# Dominance, not an absolute floor on the window-mean: the override can commit in a sharp
# late-window spike (one component near-zero for most of 17-22, then jumping at the commit
# layer) — that shape pulls its window-mean down without making it any less the writer. So
# decide on which component dominates the other, corroborated by which one peaks higher.
DOMINANCE = 2.0
if attn_win >= DOMINANCE * mlp_win and ma[peak_layer_attn] >= mm[peak_layer_mlp]:
    PHASE_A, WRITER = "attention-written", "attn"
elif mlp_win >= DOMINANCE * attn_win and mm[peak_layer_mlp] >= ma[peak_layer_attn]:
    PHASE_A, WRITER = "MLP-written", "mlp"
else:
    PHASE_A, WRITER = "mixed", ("attn" if attn_win >= mlp_win else "mlp")
loser        = "mlp" if WRITER == "attn" else "attn"
HEAD_LAYER   = peak_layer_attn                                  # heads decompose attn_out only
COMMIT_LAYER = peak_layer_attn if WRITER == "attn" else peak_layer_mlp
attn_share   = attn_win / (attn_win + mlp_win + 1e-9)
print(f"\n>>> PHASE A DECISION: {PHASE_A.upper()}  "
      f"(writer={WRITER}, commit layer L{COMMIT_LAYER}, attn share of window recovery = {attn_share:.0%})")

## 6 · Phase B — heads

Per-head recovery at the attention peak layer, ranked. This decomposes the attention write specifically, so if Phase A came back MLP-written it's exploratory — we run it anyway. `TOP_HEADS` is the top three, reused in C and the redirect.

In [ ]:
if WRITER != "attn":
    print(f"NOTE: Phase A = {PHASE_A}; head decomposition is attention-specific (exploratory here).")
head_arr  = np.array([run_head_sweep(c, HEAD_LAYER) for c in FULL])   # [n_items, N_HEADS]
head_mean = head_arr.mean(0)
order     = np.argsort(head_mean)[::-1]
TOP_HEADS = [int(h) for h in order[:3]]

plt.figure(figsize=(8, 0.42 * N_HEADS + 1))
plt.barh(range(N_HEADS), head_mean[order][::-1])
plt.yticks(range(N_HEADS), [f"L{HEAD_LAYER}.H{h}" for h in order][::-1])
plt.axvline(0, lw=.8, c="gray")
plt.xlabel("mean normalized recovery (single-head patch)")
plt.title(f"Phase B — per-head recovery at L{HEAD_LAYER}")
plt.tight_layout(); plt.savefig("fig_phaseB_heads.png", dpi=130); plt.show()

print(f"head ranking @L{HEAD_LAYER}:")
for h in order: print(f"  H{h:>2}: {head_mean[h]:+.3f}")
print(f"\nTOP_HEADS = {TOP_HEADS}  recoveries={[round(float(head_mean[h]), 3) for h in TOP_HEADS]}")

## 7 · Phase C — sufficiency + necessity

Sufficiency: patch the top-3 heads alone into the clean run and report the flip-to-target rate (how often `patched_ld > 0`) and mean recovery. Necessity: at the commit layer, contrast winner-alone against loser-alone recovery.

In [ ]:
suff = []
for c in FULL:
    donor = c["store"]["ohead"][HEAD_LAYER]
    ld = patch_heads_ld(c["c_prefix"], HEAD_LAYER, TOP_HEADS, donor, c["tgt"], c["base"])
    suff.append(dict(ld=ld, rec=recovery(ld, c["c_ld"], c["i_ld"]), flipped=bool(ld > 0)))
flip_rate = float(np.mean([r["flipped"] for r in suff]))
mean_rec  = float(np.mean([r["rec"] for r in suff]))
print(f"SUFFICIENCY (top-3 heads {TOP_HEADS} @L{HEAD_LAYER}, n={len(FULL)}):")
print(f"  flip-to-target rate (patched_ld > 0) = {flip_rate:.2f}")
print(f"  mean recovery                        = {mean_rec:+.2f}")

win_recs, los_recs = [], []
for c in FULL:
    win_recs.append(recovery(patch_component_ld(c["c_prefix"], COMMIT_LAYER, WRITER,
                    c["store"][WRITER][COMMIT_LAYER], c["tgt"], c["base"]), c["c_ld"], c["i_ld"]))
    los_recs.append(recovery(patch_component_ld(c["c_prefix"], COMMIT_LAYER, loser,
                    c["store"][loser][COMMIT_LAYER], c["tgt"], c["base"]), c["c_ld"], c["i_ld"]))
win_m, los_m = float(np.mean(win_recs)), float(np.mean(los_recs))
print(f"\nNECESSITY contrast @L{COMMIT_LAYER} (n={len(FULL)}):")
print(f"  {WRITER}-alone recovery = {win_m:+.2f}")
print(f"  {loser}-alone recovery = {los_m:+.2f}")
print(f"  contrast ({WRITER} − {loser}) = {win_m - los_m:+.2f}")

## 8 · Stretch — redirect to a letter we pick

Find pairs (A, B) where A's target letter X shows up as a distractor in B (B's own target is something else, and X isn't B's clean answer). Transplant the top heads' o_proj outputs from A's X-targeting injected run into B's clean run, and check whether B now commits to X. Report the flip-to-X rate. Small and careful.

In [ ]:
def letter_options(item):
    return set(CONFIG["LETTERS"][:len(item.options)])

items_by_key = {item_key(it): it for it in items}
pairs = []
for ca in FULL:
    X = ca["tgt"]
    for cb in FULL:
        if cb["key"] == ca["key"]: continue
        B = items_by_key[cb["key"]]
        if X in letter_options(B) and X != cb["tgt"] and X != cb["base"]:
            pairs.append((ca, cb, X))
print(f"redirect candidate pairs: {len(pairs)}")

MAXP, redir = 40, []
for ca, cb, X in pairs[:MAXP]:
    donor = ca["store"]["ohead"][HEAD_LAYER]        # heads captured on A's X-targeting injected run
    def pre_hook(m, inp, donor=donor):
        x = inp[0].clone()
        for h in TOP_HEADS:
            sl = slice(h*HEAD_DIM, (h+1)*HEAD_DIM)
            x[:, -1, sl] = donor[:, sl].to(x.dtype)
        return (x,) + tuple(inp[1:])
    handle = _layers()[HEAD_LAYER].self_attn.o_proj.register_forward_pre_hook(pre_hook)
    try:
        with torch.inference_mode():
            logits = model(cb["c_prefix"]).logits[:, -1, :]
    finally:
        handle.remove()
    redir.append(letter_argmax(logits) == X)
redirect_rate = float(np.mean(redir)) if redir else float("nan")
print(f"REDIRECT flip-to-chosen-letter rate (B argmax == X): {redirect_rate:.2f}  (n={len(redir)} pairs)")

## 9 · Phase D — overwrite vs suppress

Bullet-2's first question: when the injection flips the answer, does it *erase* the model's original answer or just *outvote* it? Read the model's internal belief in each letter with a logit lens at the answer position — project every layer's residual through the final norm and unembedding — for the clean and injected runs.

The injection's swing in target-minus-baseline logit space splits cleanly in two: how much it *adds* to the target, and how much it *takes from* the original answer. `overwrite_share` is the baseline-destruction fraction — near 1 means the override mostly tears the original belief down (overwrite), near 0 means the original survives and is simply outvoted (suppress). The layer plot shows where that happens relative to the 17–22 window.

In [ ]:
# --- Phase D: overwrite vs suppress -------------------------------------------
# When the injection flips the answer, is the model's ORIGINAL answer erased (overwrite)
# or left intact but outvoted (suppress)? Logit-lens the answer position at every layer
# — project each layer's residual through the final norm + unembedding — for the clean
# and injected runs, and read the baseline (originally-correct) and target letters.
assert globals().get("VALIDATION_OK"), "run Validation + Phases A/B/C first"

_letter_ids = torch.tensor([LETTER_ID[L] for L in CONFIG["LETTERS"]], device=CONFIG["DEVICE"])
W_letters   = model.lm_head.weight[_letter_ids]                 # [4, d_model] unembed rows
L2I         = {L: i for i, L in enumerate(CONFIG["LETTERS"])}

@torch.inference_mode()
def lens_letter_logits(resid_vec):
    # resid_vec: [1, d_model] layer output at the answer position -> 4 letter logits.
    h = model.model.norm(resid_vec.to(W_letters.dtype))
    return (h @ W_letters.t())[0].float()

def belief_trajectories(ctx):
    ti, bi = L2I[ctx["tgt"]], L2I[ctx["base"]]
    clean_resid = capture_injected(ctx["c_prefix"], pos=-1)["resid"]
    inj_resid   = ctx["store"]["resid"]
    cb, ct, ib, it = [], [], [], []
    for L in range(LAYERS):
        cl, il = lens_letter_logits(clean_resid[L]), lens_letter_logits(inj_resid[L])
        cb.append(float(cl[bi])); ct.append(float(cl[ti]))
        ib.append(float(il[bi])); it.append(float(il[ti]))
    return cb, ct, ib, it

CB, CT, IB, IT = [], [], [], []
for c in FULL:
    cb, ct, ib, it = belief_trajectories(c)
    CB.append(cb); CT.append(ct); IB.append(ib); IT.append(it)
CB, CT, IB, IT = (np.array(x) for x in (CB, CT, IB, IT))        # each [n_items, LAYERS]

# sanity: the lens at the final layer should reproduce the real readout logit-diff
lens_ld = float(np.mean(IT[:, -1] - IB[:, -1]))
real_ld = float(np.mean([c["i_ld"] for c in FULL]))
print(f"lens sanity: last-layer lens ld={lens_ld:+.2f} vs real injected ld={real_ld:+.2f} (should match)")

# decompose the readout swing per item: swing = (target gained) + (baseline lost)
dT = IT[:, -1] - CT[:, -1]                                      # target gained by injection
dB = IB[:, -1] - CB[:, -1]                                      # baseline change (neg = lost)
swing = dT - dB
overwrite_share = np.clip(-dB, 0, None) / np.clip(swing, 1e-6, None)
ow = float(np.nanmean(overwrite_share))
verdict = ("overwrite-dominant" if ow >= 0.66 else
           "suppression-dominant" if ow <= 0.34 else "mixed (both mechanisms)")
print(f"\nPHASE D — overwrite vs suppress (n={len(FULL)}):")
print(f"  mean target gained   (+dT) = {float(np.nanmean(dT)):+.2f}")
print(f"  mean baseline lost   (-dB) = {float(np.nanmean(-dB)):+.2f}")
print(f"  overwrite_share            = {ow:.2f}  ->  {verdict}")
print(f"  (~1 = injection erases the original answer; ~0 = original survives, just outvoted)")

# layer-wise: does the baseline belief collapse, or hold while the target climbs past it?
mcb, mct, mib, mit = CB.mean(0), CT.mean(0), IB.mean(0), IT.mean(0)
plt.figure(figsize=(10, 5))
xs = range(LAYERS)
plt.plot(xs, mcb, color="C0", ls="--", label="baseline letter — clean")
plt.plot(xs, mib, color="C0",          label="baseline letter — injected")
plt.plot(xs, mct, color="C1", ls="--", label="target letter — clean")
plt.plot(xs, mit, color="C1",          label="target letter — injected")
plt.axvspan(CONFIG["WINDOW"][0], CONFIG["WINDOW"][1], alpha=0.08, color="red")
plt.xlabel("layer"); plt.ylabel("logit-lens logit at answer position")
plt.title("Phase D — internal belief: baseline vs target, clean vs injected")
plt.legend(); plt.tight_layout(); plt.savefig("fig_phaseD_overwrite_suppress.png", dpi=130); plt.show()

## 10 · Phase E — text-monitor blindness

Bullet-2's second claim. Phase B/C reduced the override to a few heads at the peak attention layer. Those heads can be driven directly: patch their o_proj outputs into a *clean* run — no injected `<think>` text at all — and the answer still flips. But the trace a CoT monitor reads is the clean prefix, byte-identical to the un-manipulated run, so any monitor that's a function of the text returns the same verdict on both and can't see the edit.

This is the precise (and partly definitional) sense of "blind to mid/late-layer state manipulation": the same override a text injection installs can instead be installed as a small activation edit at the peak layer, flipping the answer while leaving nothing for a text monitor to flag. It matters because it shares the injection study's threat model — an attacker controlling the open-weight runtime — and the activation form is strictly stealthier. The cell asserts the monitored text is unchanged, reports the flip rate, and can call a live monitor if `OPENROUTER_API_KEY` is set; the structural result holds regardless.

In [ ]:
# --- Phase E: text-based monitoring is blind to the mid/late-layer edit --------
# The override reduces to the top heads at HEAD_LAYER (Phase B/C). Drive them directly:
# patch their o_proj outputs into a CLEAN run (no injected <think> text). The answer
# flips, but the prompt a CoT monitor reads is the clean prefix — identical to the
# un-manipulated run — so a text-only monitor cannot tell the two apart.
assert globals().get("TOP_HEADS") is not None, "run Phase B/C first"

@torch.inference_mode()
def answer_under_head_patch(ctx, heads, layer):
    donor = ctx["store"]["ohead"][layer]
    def pre_hook(m, inp):
        x = inp[0].clone()
        for h in heads:
            sl = slice(h*HEAD_DIM, (h+1)*HEAD_DIM)
            x[:, -1, sl] = donor[:, sl].to(x.dtype)
        return (x,) + tuple(inp[1:])
    handle = _layers()[layer].self_attn.o_proj.register_forward_pre_hook(pre_hook)
    try:
        return letter_argmax(model(ctx["c_prefix"]).logits[:, -1, :])
    finally:
        handle.remove()

rows, flips = [], 0
for c in FULL:
    patched_ans = answer_under_head_patch(c, TOP_HEADS, HEAD_LAYER)
    flipped = (patched_ans == c["tgt"])                        # FULL is capturable, so clean != tgt
    flips += int(flipped)
    txt = tokenizer.decode(c["c_prefix"][0], skip_special_tokens=True)   # what a monitor would read
    rows.append(dict(clean=c["c_letter"], patched=patched_ans, flipped=flipped,
                     text_identical=True))                     # patch is in activations; text unchanged
flip_rate_E = flips / len(FULL)
assert all(r["text_identical"] for r in rows)
print(f"PHASE E — monitor blindness (n={len(FULL)}, heads {TOP_HEADS} @L{HEAD_LAYER}):")
print(f"  answer flipped to target by the activation edit : {flip_rate_E:.0%} of items")
print(f"  monitor-visible <think> text changed by the edit: 0 of {len(FULL)} items")
print(f"  => any text-only CoT monitor returns identical verdicts on edited vs clean runs;")
print(f"     its detection rate for this manipulation is 0 by construction.")

import os
if os.environ.get("OPENROUTER_API_KEY"):
    print("\n[optional] OPENROUTER_API_KEY found — run the repo's monitor stage on these clean")
    print("traces; the edited-run text is identical, so the flags come back identical too.")
else:
    print("\n[optional] set OPENROUTER_API_KEY to run a live monitor; the structural result")
    print("holds regardless (identical text in -> identical flag out).")

# fold the mechanism findings into findings.txt (appends; the original blurb cell is untouched)
_extra = (
    "\n\nMECHANISM EXTENSIONS\n"
    f"Phase D — {verdict}. Of the injection's target-minus-baseline swing at the readout, "
    f"{float(np.nanmean(-dB)):+.2f} comes from tearing down the model's original answer and "
    f"{float(np.nanmean(dT)):+.2f} from adding target evidence (overwrite_share={ow:.2f}); "
    f"the override {'largely overwrites' if ow>=0.66 else 'largely suppresses' if ow<=0.34 else 'both overwrites and suppresses'} the prior belief.\n"
    f"Phase E — the override reduces to an activation edit on {len(TOP_HEADS)} heads at L{HEAD_LAYER}: "
    f"patched into a clean run it flips the answer in {flip_rate_E:.0%} of items while leaving the "
    f"<think> trace byte-identical, so a text-based CoT monitor is blind to it by construction.\n"
)
with open("findings.txt", "a") as fh:
    fh.write(_extra)
print("\nappended mechanism findings to findings.txt")

## 11 · Results summary

A short blurb filled in for whichever branch Phase A took, plus the key numbers, written to `findings.txt` for the writeup.

In [ ]:
def fmt(x): return f"{x:+.2f}"
mech = ("attention-mediated (induction/retrieval-head copying of the planted conclusion; Olsson et al. 2022)"
        if WRITER == "attn" else
        "MLP-mediated (a mid-layer factual association being overwritten; Meng et al. ROME 2022)")
win_rec = attn_win if WRITER == "attn" else mlp_win
los_rec = mlp_win  if WRITER == "attn" else attn_win

blurb = f'''COMPONENT-RESOLUTION FINDINGS  ({len(FULL)} effect-substantial items; metric = normalized
recovery in target-minus-baseline logit space; bootstrap {CONFIG["N_BOOT"]} resamples, seed 0).

Phase A — {PHASE_A}. Across the parent localization window (layers {CONFIG["WINDOW"][0]}-{CONFIG["WINDOW"][1]}),
the {WRITER}_out marginal write recovers {fmt(win_rec)} of the injection effect versus {fmt(los_rec)} for
the other sublayer, so the override is {mech}, committing at layer L{COMMIT_LAYER}.

Phase B/C — at L{HEAD_LAYER} the top heads are {TOP_HEADS}. Patching these three alone into the clean run
flips the answer to the target in {flip_rate:.0%} of items (mean recovery {fmt(mean_rec)}); the necessity
contrast at the commit layer is {WRITER}-alone {fmt(win_m)} vs {loser}-alone {fmt(los_m)}. Redirect:
transplanting these heads from an X-targeting run into a different item drives that item to X in
{redirect_rate:.0%} of tested pairs.'''
print(blurb)
open("findings.txt", "w").write(blurb)

## 12 · Persist outputs

Save the recovery arrays (`.npz`) and figures (`.png`) — Colab is ephemeral — and copy to Drive if it's mounted.

In [ ]:
np.savez("component_resolution_arrays.npz",
         layers=np.arange(LAYERS),
         full_resid=full_curves, full_resid_mean=m_full,
         attn=attn_arr, mlp=mlp_arr, attn_mean=ma, mlp_mean=mm,
         heads=head_arr, head_mean=head_mean,
         head_layer=HEAD_LAYER, commit_layer=COMMIT_LAYER, top_heads=np.array(TOP_HEADS),
         attn_win=attn_win, mlp_win=mlp_win,
         suff_flip_rate=flip_rate, suff_mean_rec=mean_rec,
         necessity_winner=win_m, necessity_loser=los_m,
         redirect_rate=redirect_rate, window=np.array(CONFIG["WINDOW"]),
         decision=np.array(PHASE_A), writer=np.array(WRITER))
figs = ["fig_validation_fullresid.png", "fig_phaseA_attn_vs_mlp.png", "fig_phaseB_heads.png"]
print("saved:", "component_resolution_arrays.npz", "findings.txt", *figs)

import shutil, os
for base in ("/content/drive/MyDrive", "/gdrive/MyDrive"):
    if os.path.isdir(base):
        dest = os.path.join(base, "cot-injection-monitoring", "component_resolution")
        os.makedirs(dest, exist_ok=True)
        for f in ["component_resolution_arrays.npz", "findings.txt", PREP_FILE, *figs]:
            if os.path.exists(f): shutil.copy(f, dest)
        print("copied outputs to", dest); break
else:
    print("no Drive mount found; outputs saved locally only")

---
### Reading this notebook
- Validation reproduces the 17–21 rise → harness, loading and hooks are sound, so the component curves are trustworthy.
- The Phase A call is the headline: attention-written ⇒ retrieval/induction heads copy the planted conclusion; MLP-written ⇒ a mid-layer association is overwritten; mixed ⇒ both.
- B and C name the heads and show they're sufficient (patch alone → flip) and that the winner dominates the loser at the commit layer.
- Redirect is the strongest causal check: the same heads, moved into a new item, steer it to an answer we chose — so the override is a reusable, head-localized mechanism, not an item-specific fluke.
- Phase D answers overwrite vs suppress: it logit-lenses the original answer's internal trajectory and splits the swing into baseline-destruction vs target-addition.
- Phase E shows the override can be installed as a small activation edit at the peak heads that flips the answer while leaving the monitored text unchanged — a text-based CoT monitor is blind to it.